### **Portefólio AR — Demonstração unificada Tic-Tac-Toe**

Um *walkthrough* executável que treina e avalia, na mesma sessão e com a mesma função de avaliação, os principais agentes do portefólio sobre o mesmo ambiente:
**Aleatório → SARSA → Q-Learning → REINFORCE (sem baseline) → REINFORCE + baseline → MCTS**.

Práticas **PL6/PL7** (ambiente + features + SARSA/Q-Learning), **PL8** (REINFORCE), **PL9** (Monte Carlo Tree Search).


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import matplotlib.pyplot as plt

from AR1.envs.tictactoe import TicTacToeEnv, _winner
from AR1.policies.tictactoe import random_action

SEED        = 7
N_EVAL      = 500    # jogos de avaliação por lado
N_TRAIN     = 5_000  # episódios de treino para SARSA / Q-Learning
N_REINFORCE = 4_000  # episódios de treino para REINFORCE

np.random.seed(SEED)
print(f"seed = {SEED}   jogos de avaliação = {N_EVAL}   episódios de treino = {N_TRAIN}")


---
#### **1. Helper de avaliação partilhado**

Cada agente abaixo é exposto como um *callable* `policy(env, state) → action`. Este *helper* único joga `n_games` jogos contra um adversário aleatório uniforme e devolve as taxas de vitória / empate / derrota. Corremos sempre duas vezes por agente: uma como **X** (primeiro a jogar) e outra como **O** (segundo a jogar).

A mesma *seed* é reutilizada entre agentes para que qualquer diferença observada **não** seja atribuível ao acaso.


In [ ]:
def evaluate_vs_random(policy_fn, n_games=N_EVAL, as_player=1, seed=SEED):
    env = TicTacToeEnv()
    wins = draws = losses = 0
    for _ in range(n_games):
        state = env.reset()
        done = False
        while not done:
            if env.current_player == as_player:
                action = policy_fn(env, state)
            else:
                action = random_action(env, state)
            state, _, done = env.step(action)
        w = _winner(state)
        if w == as_player:
            wins += 1
        elif w == 0:
            draws += 1
        else:
            losses += 1
    return wins / n_games, draws / n_games, losses / n_games


def both_sides(policy_fn, label, seed=SEED):
    wx, dx, lx = evaluate_vs_random(policy_fn, as_player=1,  seed=seed)
    wo, do, lo = evaluate_vs_random(policy_fn, as_player=-1, seed=seed + 1)
    print(f"{label:<22}  X: win={wx:.1%} draw={dx:.1%} loss={lx:.1%}  |  "
          f"O: win={wo:.1%} draw={do:.1%} loss={lo:.1%}")
    return {"as_X": (wx, dx, lx), "as_O": (wo, do, lo)}


results = {}


---
#### **2. Baseline aleatório**

Dois jogadores uniformemente aleatórios — é o piso da dificuldade e a referência para tudo o que se segue. O X tem uma vantagem estrutural (primeiro a jogar) que já empurra a taxa de vitória bem acima dos 50 %.


In [ ]:
def random_policy(env, state):
    return random_action(env, state)

results["random"] = both_sides(random_policy, "Random")


---
#### **3. Controlo tabular com mascaramento de ações ilegais (PL6/PL7)**

`TicTacToeSarsa` e `TicTacToeQLearning` (em `scripts/run_tictactoe.py`) são agentes tabulares cujo `select_action(state, available)` ignora células ilegais. São treinados contra um adversário aleatório e avaliados de forma *greedy*.

**SARSA (*on-policy*):** o Q é atualizado com a próxima ação que vai *de facto* ser tomada — fica ligeiramente conservador porque a exploração faz parte da política que está a ser avaliada.

$$Q(s,a) \leftarrow Q(s,a) + \alpha \,[\, r + \gamma\, Q(s', a') - Q(s, a)\,]$$

**Q-Learning (*off-policy*):** o Q é atualizado com a *melhor* próxima ação, independentemente do que é jogado — converge para a política ótima sob condições razoáveis.

$$Q(s,a) \leftarrow Q(s,a) + \alpha \,[\, r + \gamma\, \max_{a'} Q(s', a') - Q(s, a)\,]$$


In [ ]:
from AR1.scripts.run_tictactoe import TicTacToeSarsa, TicTacToeQLearning, train_agent

def _train_and_wrap(AgentCls, episodes=N_TRAIN, seed=SEED):
    agent = AgentCls(epsilon=0.1, alpha=0.5, gamma=1.0, seed=seed)
    train_agent(agent, num_episodes=episodes, seed=seed)

    def policy(env, state):
        avail = env.available_actions(state)
        return agent.greedy_action(state, avail)
    return agent, policy

print("A treinar SARSA ...")
sarsa_agent, sarsa_policy = _train_and_wrap(TicTacToeSarsa)
results["sarsa"] = both_sides(sarsa_policy, "SARSA")

print("\nA treinar Q-Learning ...")
ql_agent, ql_policy = _train_and_wrap(TicTacToeQLearning)
results["q_learning"] = both_sides(ql_policy, "Q-Learning")


---
#### **4. REINFORCE — gradiente de política Monte Carlo (PL8)**

Uma **política softmax linear** no encoding 27-dim relativo ao jogador ($\theta \in \mathbb{R}^{9 \times 27}$):

$$\pi(a\,|\,s) = \frac{e^{\theta_a \cdot \phi(s)}}{\sum_{b \in \mathcal{A}(s)} e^{\theta_b \cdot \phi(s)}}$$

**Atualização vanilla** (um passo por episódio, $G_t$ = retorno Monte Carlo):

$$\theta \leftarrow \theta + \alpha \,\gamma^t\, G_t\, \nabla_\theta \log \pi(a_t\,|\,s_t)$$

**Baseline (Sutton & Barto, Sec. 13.4)** subtrai um $V(s_t) = w \cdot \phi(s_t)$ aprendido aos retornos, reduzindo a variância **sem** introduzir *bias*:

$$\theta \leftarrow \theta + \alpha \,\gamma^t\, [\,G_t - V(s_t)\,]\, \nabla_\theta \log \pi(a_t\,|\,s_t)$$

Treinamos ambas as variantes em *self-play* com 30 % dos episódios substituídos por adversário aleatório (ajuda a política a manter-se afiada contra jogo fraco).


In [ ]:
from AR1.agents.control.reinforce import ReinforceAgent
from AR1.experiments.reinforce_tictactoe import (
    run_reinforce_episode,
    run_vs_random_episode,
)
from AR1.features.tictactoe import encode_state

def train_reinforce(agent, episodes=N_REINFORCE, random_frac=0.3, seed=SEED):
    rng = np.random.default_rng(seed)
    env = TicTacToeEnv()
    for _ in range(episodes):
        if rng.random() < random_frac:
            run_vs_random_episode(env, agent)
        else:
            run_reinforce_episode(env, agent)
    return agent

def reinforce_policy_for(agent):
    def policy(env, state):
        phi   = encode_state(state, env.current_player)
        avail = env.available_actions(state)
        return agent.greedy_action(phi, avail)
    return policy

print("A treinar REINFORCE (vanilla) ...")
ra = train_reinforce(ReinforceAgent(alpha=0.01, use_baseline=False, seed=SEED))
results["reinforce_vanilla"] = both_sides(reinforce_policy_for(ra), "REINFORCE vanilla")

print("\nA treinar REINFORCE + baseline ...")
rb = train_reinforce(ReinforceAgent(alpha=0.01, use_baseline=True, seed=SEED + 1))
results["reinforce_baseline"] = both_sides(reinforce_policy_for(rb), "REINFORCE + baseline")


---
#### **5. Monte Carlo Tree Search — planeamento model-based (PL9)**

O MCTS é o único agente aqui que **não aprende nada**. A cada jogada constrói uma árvore de procura de raiz, executando $N$ simulações de quatro fases:

1. **Seleção** — descer com UCB1 até encontrar um nó que não esteja totalmente expandido ou seja terminal:
$$\text{UCB}(s,a) = Q(s,a) + c\,\sqrt{\frac{\ln N(s)}{N(s,a)}}$$
2. **Expansão** — adicionar um novo filho para uma ação ainda não testada.
3. **Simulação** — *rollout* aleatório até ao fim do jogo.
4. **Backup** — propagar o resultado pela árvore, **invertendo o sinal a cada nível** (o pai é o adversário).

Ação escolhida = **filho mais visitado da raiz** (robust best — menos sensível a *outliers* dos rollouts do que $\arg\max Q$).


In [ ]:
from AR1.agents.planning.mcts import MCTSAgent

mcts = MCTSAgent(n_simulations=200)

def mcts_policy(env, state):
    return mcts.select_action(state, env.current_player)

results["mcts_200"] = both_sides(mcts_policy, "MCTS (200 sims)")


---
#### **6. Gráfico comparativo**

Taxas de **vitória / empate / derrota** empilhadas contra o adversário aleatório — painel esquerdo: agente como **X** (primeiro a jogar); painel direito: como **O** (segundo a jogar). A hierarquia é o *punchline* do portefólio:

| Capacidade adicionada | Algoritmo | Win-rate típico como X |
|---|---|---:|
| _(nenhuma)_                            | Aleatório              | ~58 % |
| Controlo tabular, off-policy           | Q-Learning             | ~98 % |
| Controlo tabular, on-policy            | SARSA                  | ~98 % |
| Gradiente de política + redução var.   | REINFORCE + baseline   | ~96 % |
| Gradiente de política (vanilla)        | REINFORCE              | ~78 % |
| Planeamento em tempo de decisão        | MCTS                   | **100 %** |


In [ ]:
def stacked_bars(ax, role, title):
    labels = list(results.keys())
    wins   = [results[k][role][0] for k in labels]
    draws  = [results[k][role][1] for k in labels]
    losses = [results[k][role][2] for k in labels]
    x = np.arange(len(labels))
    ax.bar(x, wins,                                                  label="vitória", color="#4caf50")
    ax.bar(x, draws,  bottom=wins,                                   label="empate",  color="#ffb300")
    ax.bar(x, losses, bottom=np.array(wins) + np.array(draws),       label="derrota", color="#e53935")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("fração de jogos")
    ax.set_title(title)
    ax.grid(alpha=0.2, axis="y")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
stacked_bars(axes[0], "as_X", "como X (primeiro a jogar)")
stacked_bars(axes[1], "as_O", "como O (segundo a jogar)")
axes[1].legend(loc="upper right")
fig.suptitle("Tic-Tac-Toe — agentes contra adversário aleatório")
fig.tight_layout()
plt.show()


---
#### **7. Discussão**

A tabela acima fala por si: cada **capacidade** acrescentada melhora a *win rate*.

* O **aleatório** ganha ~58 % como X só por ser o primeiro a jogar — é a vantagem estrutural do tabuleiro.
* **SARSA e Q-Learning** sobem para ~98 % porque aprendem uma função de valor *Q(s, a)* sobre estados visitados. O **Q-Learning** é ligeiramente melhor porque é *off-policy*: aprende a política ótima mesmo enquanto explora.
* O **REINFORCE vanilla** sofre com a variância dos retornos *G<sub>t</sub>* — sem baseline, o gradiente é ruidoso e a política converge mal. **Adicionar V(s)** como baseline reduz a variância sem introduzir *bias* e passa de ~78 % para ~96 %.
* O **MCTS** vence sempre que joga primeiro porque o horizonte do Tic-Tac-Toe é tão curto (≤ 9 jogadas) que 200 simulações chegam para explorar a sub-árvore relevante. Não há treino — é planeamento em tempo de decisão.
* **Como O** todas as taxas descem: jogar segundo obriga a reagir em vez de ditar.

> Detalhes técnicos completos (incluindo as extensões DQN e AlphaZero treinadas neste ambiente): [`RESULTS.md`](../RESULTS.md).


In [ ]:
# Resumo numerico com os resultados reais obtidos neste notebook
print(f"{'Agente':<22} | {'X win':>7} | {'X empate':>9} | {'X derrota':>10} | {'O win':>7} | {'O empate':>9} | {'O derrota':>10}")
print('-' * 96)
for name, r in results.items():
    wx, dx, lx = r['as_X']
    wo, do, lo = r['as_O']
    print(f"{name:<22} | {wx:>6.1%} | {dx:>8.1%} | {lx:>9.1%} | {wo:>6.1%} | {do:>8.1%} | {lo:>9.1%}")


---

*Para reproduzir todas as outras experiências do portefólio (PL1-PL9 + DQN + AlphaZero), correr:*

```bash
python -m AR1.scripts.run_benchmarks --no-show
```
